In [56]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Generar dataset

np.random.seed(42)
n = 500
data = pd.DataFrame({
    "edad": np.random.randint(18, 65, size=n),
    "visitas": np.random.poisson(10, size=n),
    "tiempo_sesion": np.random.normal(5, 2, size=n).clip(1, 15),
    "pais": np.random.choice(["Chile", "Argentina", "Peru", "Mexico"], size=n),
    "dispositivo": np.random.choice(["Desktop", "Mobile", "Tablet"], size=n),
    "monto_compra": np.random.normal(200, 50, size=n) \
                    + 2*np.random.randint(18,65,size=n) \
                    + 5*np.random.poisson(10, size=n)
})
data.to_csv("clientes.csv", index=False)

X = data.drop("monto_compra", axis=1)
y = data["monto_compra"]

num_cols = ["edad","visitas","tiempo_sesion"]
cat_cols = ["pais","dispositivo"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ● Lección 1: Fundamentos del Aprendizaje de Máquina


In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("clientes.csv")
X = df.drop("monto_compra", axis=1)
y = df["monto_compra"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X.shape, X_train.shape, X_test.shape)


(500, 5) (400, 5) (100, 5)


# ● Lección 2: Nivel de ajuste del modelo y validación cruzada



In [68]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline

# Create a pipeline that first preprocesses the data and then applies Linear Regression
model_pipeline = make_pipeline(preprocessor, LinearRegression())

model_pipeline.fit(X_train, y_train)
y_pred_train = model_pipeline.predict(X_train)
y_pred_test = model_pipeline.predict(X_test)

mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = -cross_val_score(model_pipeline, X, y, cv=kf, scoring="neg_mean_absolute_error")
print("MAE train", mae_train, "MAE test", mae_test, "CV MAE mean", cv_scores.mean())

MAE train 43.34668412450012 MAE test 47.12232267057553 CV MAE mean 44.84802570823538


#● Lección 3: Preprocesamiento y escalamiento de datos


In [59]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object","category"]).columns.tolist()

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

full_pipeline = Pipeline([
    ("preprocessor", preprocessor)
])
X_train_prep = full_pipeline.fit_transform(X_train)


# ● Lección 4: Regresiones

In [60]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

lin = make_pipeline(preprocessor, LinearRegression())
lin.fit(X_train, y_train)
y_pred = lin.predict(X_test)
print("R2 linear", r2_score(y_test, y_pred))

poly = make_pipeline(preprocessor, PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
poly.fit(X_train, y_train)
print("R2 poly", r2_score(y_test, poly.predict(X_test)))


R2 linear -0.03210401422184539
R2 poly -0.21569178476357243


# ● Lección 5: Algoritmos de clasificación

In [61]:
from sklearn.neighbors import KNeighborsClassifier
# crear etiquetas simuladas
y_cat = pd.qcut(y, q=3, labels=["bajo","medio","alto"])
knn = make_pipeline(preprocessor, KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_cat.loc[X_train.index])
print("Score KNN", knn.score(X_test, y_cat.loc[X_test.index]))


Score KNN 0.37


#● Lección 6: Métricas de desempeño

In [62]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

models = {"Linear": lin, "Poly": poly}
results = {}
for name, m in models.items():
    yhat = m.predict(X_test)
    results[name] = metrics(y_test, yhat)
import pandas as pd
pd.DataFrame(results).T


,MAE,MSE,RMSE,R2
Linear,47.122323,3193.87704,56.514397,-0.032104
Poly,50.678334,3761.99494,61.335104,-0.215692


#● Lección 7: Optimización del modelo

In [63]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge, Lasso

pipe_ridge = make_pipeline(preprocessor, Ridge())
param_grid = {"ridge__alpha": [0.1, 1.0, 10.0]}
gs = GridSearchCV(pipe_ridge, param_grid, cv=5, scoring="neg_mean_absolute_error")
gs.fit(X_train, y_train)
print("Mejor parametro", gs.best_params_, "Mejor  MAE", -gs.best_score_)


Mejor parametro {'ridge__alpha': 10.0} Mejor  MAE 44.478306283308086


In [64]:

ridge_pipe = make_pipeline(preprocessor, Ridge())
param_grid = {"ridge__alpha": [0.1, 1.0, 10.0]}
gs_ridge = GridSearchCV(ridge_pipe, param_grid, cv=5, scoring="neg_mean_absolute_error")
gs_ridge.fit(X_train, y_train)
print("Mejor Parametro Ridge :", gs_ridge.best_params_)

lasso_pipe = make_pipeline(preprocessor, Lasso())
param_grid_lasso = {"lasso__alpha": [0.001, 0.01, 0.1]}
gs_lasso = GridSearchCV(lasso_pipe, param_grid_lasso, cv=5, scoring="neg_mean_absolute_error")
gs_lasso.fit(X_train, y_train)
print("Mejor Parametro Lasso:", gs_lasso.best_params_)



Mejor Parametro Ridge : {'ridge__alpha': 10.0}
Mejor Parametro Lasso: {'lasso__alpha': 0.1}


#● Lección 8: Algoritmos de Boosting

In [65]:
from sklearn.ensemble import GradientBoostingRegressor

pipe_gb = make_pipeline(preprocessor, GradientBoostingRegressor(random_state=42))
param_grid_gb = {
    "gradientboostingregressor__n_estimators": [100, 200],
    "gradientboostingregressor__learning_rate": [0.05, 0.1],
    "gradientboostingregressor__max_depth": [3, 5]
}
gs_gb = GridSearchCV(pipe_gb, param_grid_gb, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
gs_gb.fit(X_train, y_train)
print("Mejor GB", gs_gb.best_params_, -gs_gb.best_score_)


Mejor GB {'gradientboostingregressor__learning_rate': 0.05, 'gradientboostingregressor__max_depth': 3, 'gradientboostingregressor__n_estimators': 100} 45.3342191552745


In [66]:

gb_pipe = make_pipeline(preprocessor, GradientBoostingRegressor(random_state=42))
param_grid_gb = {
    "gradientboostingregressor__n_estimators": [100, 200],
    "gradientboostingregressor__learning_rate": [0.05, 0.1],
    "gradientboostingregressor__max_depth": [3, 5]
}
gs_gb = GridSearchCV(gb_pipe, param_grid_gb, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
gs_gb.fit(X_train, y_train)
print("Mejor parametro GB:", gs_gb.best_params_)
print("GB MAE:", mean_absolute_error(y_test, gs_gb.predict(X_test)))

Mejor parametro GB: {'gradientboostingregressor__learning_rate': 0.05, 'gradientboostingregressor__max_depth': 3, 'gradientboostingregressor__n_estimators': 100}
GB MAE: 50.211831406940746
